# <center> OriGENE-Mut <center>

## <center> *Cancer gene classification using deep convolutional neural networks on epigenetic marker enrichment profiles and mutation-derived features* <center>
**Authors:**  *Marc Pielies-Avellí*, *Ming-Heng Hsiung*  *Dr. Alin S. Tomoiaga*, *Dr. Mattias Ohlsson*, *Dr. Victor Olariu*
]
**Purpose:** This notebook extends the original OriGENE workflow by adding gene-level mutation features as a second model input.

The original OriGENE preprocessing workflow should be followed first to generate the epigenetic input arrays. This notebook starts from those preprocessed OriGENE arrays and adds the mutation-feature input branch.

## <center> TABLE OF CONTENTS <center>

#### A) SECTION I: Input preparation...............................................................................................................................................
1.1 SUMMARY: From original OriGENE preprocessing to OriGENE-Mut inputs

1.2 Save the gene index from the original OriGENE preprocessing

1.3 User configuration: input and output paths

1.4 Load OriGENE arrays and align mutation features

#### B) SECTION II: OriGENE-Mut model...............................................................................................................................................
2.1 Helper functions

2.2 Inception module

2.3 OriGENE-Mut model definition

2.4 Cross-validation training

#### C) SECTION III: Run model and save outputs...............................................................................................................................................
3.1 Train OriGENE-Mut

3.2 Output files

#### D) SECTION IV: Supplementary mutation feature documentation............................................................................................................
4.1 Mutation feature overview table

4.2 Implementation notes

# <center> SECTION I: INPUT PREPARATION <center>

### <center> 1.1 SUMMARY: From original OriGENE preprocessing to OriGENE-Mut inputs <center>

<center> ____________________________________________________________________________ <center>

**1) Generate epigenetic input using the original OriGENE workflow**

Follow the original OriGENE preprocessing notebook to generate fixed-length, gene-centered epigenetic signal arrays.

Expected outputs from the original OriGENE preprocessing step:

```text
x_epi : NumPy array with shape (n_genes, 40000, n_channels)
y     : NumPy array with shape (n_genes,)
g     : NumPy array with shape (n_genes,)  # gene names; save as genes.npy
```

**2) Extract mutation features from VEP-annotated VCF files**

Mutation features can be generated using the standalone script mutation_feature_extractor.py. This script computes 28 gene-level mutation features from VEP-annotated VCF files, following the definitions used in this study.

The feature definitions, VEP consequence mapping, pseudocount handling, and recurrent missense implementation are documented in mutation_feature_extractor.py and Supplementary Table IV.

Recommended usage (reproducible pipeline):
To ensure consistency and reproducibility with the OriGENE framework, users are strongly encouraged to run the provided mutation Snakemake pipeline (mutation_pipeline/) rather than executing the script manually. The pipeline handles preprocessing, input validation, and standardized output generation.

Manual execution (optional):
If needed, the feature extraction script can be run independently as follows:

```bash
python mutation_feature_extractor.py \
    --vcf data/vcf/vep_annotated_sample.vcf.gz \
    --cds-lengths data/reference/Final_CDS_lengths.csv \
    --sample-id Colorectal_Cancer_I \
    --output-dir results/mutation_features \
    --mode corrected
```

Expected mutation feature output:

```text
results/mutation_features/mutation_features_Colorectal_Cancer_I.csv
```
The output is a gene-level CSV file containing one row per gene and 28 mutation-based features, which can be directly used as input to the OriGENE mutation branch.

**3) Align mutation features to the OriGENE gene order**

The mutation feature table is reordered according to `g`, the gene vector from the original OriGENE preprocessing. This step is mandatory.

**4) Train OriGENE-Mut**

The model uses the original OriGENE CNN branch and adds mutation-guided attention before the second set of inception modules.

### <center> 1.2 Save the gene index from the original OriGENE preprocessing <center>

<center> ____________________________________________________________________________ <center>

The original OriGENE preprocessing generates the epigenetic array and labels in a fixed gene order, but the gene order is not always saved as a separate file.

For the mutation branch, this gene-order file is required because mutation features must be aligned to the same gene index as the epigenetic input.

The required mapping is:

```text
x_epi[i]  <->  x_mut[i]  <->  y[i]  <->  genes[i]
```

If `genes.npy` already exists, skip this subsection. If it does not exist, add the following saving step to the original OriGENE preprocessing notebook at the same point where `x_epi` and `y` are saved.


In [ ]:
# Add this to the original OriGENE preprocessing notebook when x_epi and y are created.
# The variable `genes` must be appended in the same loop and under the same filtering conditions as x_epi and y.

# Example structure:
# x_epi = []
# y = []
# genes = []
#
# for gene in gene_list:
#     signal = ...
#     label = ...
#
#     x_epi.append(signal)
#     y.append(label)
#     genes.append(gene)

# Save all three arrays together.
# np.save("data/processed/origene/Colorectal_Cancer_I/x_epi.npy", np.array(x_epi))
# np.save("data/processed/origene/Colorectal_Cancer_I/y.npy", np.array(y))
# np.save("data/processed/origene/Colorectal_Cancer_I/genes.npy", np.array(genes))

# Sanity check after saving:
# print("x_epi:", np.array(x_epi).shape)
# print("y:", np.array(y).shape)
# print("genes:", np.array(genes).shape)
# print("First 5 genes:", genes[:5])

### <center> 1.3 User configuration: input and output paths <center>

Edit the paths below before running the notebook.

The paths are intentionally explicit so that another user can reproduce the same input-output structure without relying on local user-specific folders.

In [ ]:
from pathlib import Path

########################## Dataset name #################################################

# Keep the original single-dataset behavior, but allow OriGENE-code-style patient input.
# For patient-level input:
#   - single track: INPUT_TRACKS = ["Normal"] or ["Cancer"]
#   - double track: INPUT_TRACKS = ["Normal", "Cancer"]
# The arrays are concatenated along axis=2, exactly like OriGENE-code does for xtrn.

PATIENT = "Colorectal_Cancer_I"
DATASET_NAME = PATIENT
INPUT_TRACKS = ["single"]  # change to ["Normal", "Cancer"] for double-track patient input

#########################################################################################

########################## Input Paths ##################################################

# Option A: original behavior: one already-built OriGENE epigenetic array.
EPIGENETIC_ARRAY_PATH = Path("data/processed/origene/Colorectal_Cancer_I/x_epi.npy")

# Option B: patient-level behavior: one path per track/status.
# Uncomment this block and update paths when using patient-level single/double-track input.
# EPIGENETIC_ARRAY_PATH = [
#     Path("data/processed/origene/Liver_I/Normal/x_epi.npy"),
#     Path("data/processed/origene/Liver_I/Cancer/x_epi.npy"),
# ]
# INPUT_TRACKS = ["Normal", "Cancer"]
# PATIENT = "Liver_I"
# DATASET_NAME = f"{PATIENT}_{'_'.join(INPUT_TRACKS)}"

# Labels and genes must follow the same ordered gene set used by the epigenetic array(s).
LABEL_PATH = Path("data/processed/origene/Colorectal_Cancer_I/y.npy")
GENE_PATH = Path("data/processed/origene/Colorectal_Cancer_I/genes.npy")

# Output from mutation_feature_extractor.py
MUTATION_FEATURE_PATH = Path(
    "results/mutation_features/mutation_features_Colorectal_Cancer_I.csv"
)

#########################################################################################

########################## Output Path ##################################################

OUTPUT_DIR = Path("results/model_evaluation/mutation_branch") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#########################################################################################


### <center> 1.4 Load OriGENE arrays and align mutation features <center>

The function below loads the original OriGENE arrays and aligns the mutation feature matrix to the same gene order.

Mutation features are used in their original scale without standardization. Non-numeric values and missing values are coerced to zero during loading.

In [ ]:
import itertools
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import auc, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.layers import (
    AveragePooling1D,
    Conv1D,
    Dense,
    Dropout,
    Flatten,
    Input,
    MaxPooling1D,
    Multiply,
    concatenate,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def _load_one_epigenetic_array(path):
    """
    Load one OriGENE epigenetic array and enforce CNN-compatible shape.

    Accepted input shapes:
        (n_genes, sequence_length)      -> converted to (n_genes, sequence_length, 1)
        (n_genes, sequence_length, c)   -> kept unchanged
    """
    arr = np.load(path)

    if arr.ndim == 2:
        arr = arr.reshape((arr.shape[0], arr.shape[1], 1))
    elif arr.ndim != 3:
        raise ValueError(
            f"Epigenetic array at {path} must be 2D or 3D. Got shape {arr.shape}."
        )

    return arr.astype(np.float32)


def load_epigenetic_input(epigenetic_array_path):
    """
    Load OriGENE epigenetic input.

    This keeps the original behavior when `epigenetic_array_path` is a single Path.
    If a list/tuple of paths is provided, each array is loaded as one track/status and
    concatenated along axis=2, matching the original OriGENE-code line:

        xtrn = np.concatenate(tuple(xtrn_dict.values()), axis=2)

    Therefore:
        one path  -> single-track input, e.g. (n, 40000, 1)
        two paths -> double-track input, e.g. (n, 40000, 2)
    """
    if isinstance(epigenetic_array_path, (list, tuple)):
        arrays = [_load_one_epigenetic_array(path) for path in epigenetic_array_path]

        n_genes = {arr.shape[0] for arr in arrays}
        seq_len = {arr.shape[1] for arr in arrays}
        if len(n_genes) != 1 or len(seq_len) != 1:
            raise ValueError(
                "All epigenetic tracks must have the same number of genes and sequence length. "
                f"Got shapes: {[arr.shape for arr in arrays]}"
            )

        x_epi = np.concatenate(tuple(arrays), axis=2)
    else:
        x_epi = _load_one_epigenetic_array(epigenetic_array_path)

    return x_epi.astype(np.float32)


def load_origene_mutation_inputs(
    epigenetic_array_path,
    label_path,
    gene_path,
    mutation_feature_path,
):
    """
    Load OriGENE epigenetic inputs and align mutation features to the same gene order.

    `epigenetic_array_path` can be either:
        - a single Path to x_epi.npy, preserving the original mutation-branch workflow; or
        - a list/tuple of Paths, one per patient-level track/status, which will be
          concatenated along the channel axis like OriGENE-code.
    """

    x_epi = load_epigenetic_input(epigenetic_array_path)
    y = np.load(label_path)
    g = np.load(gene_path, allow_pickle=True)

    mut_df = pd.read_csv(mutation_feature_path)
    mut_df["Gene"] = mut_df["Gene"].astype(str).str.upper().str.strip()
    mut_df = mut_df.set_index("Gene")

    # Mutation features are used in their original scale.
    # Non-numeric values and missing values are converted to 0.
    mut_df = mut_df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    g_upper = np.array([str(gene).upper().strip() for gene in g])

    missing_genes = sorted(set(g_upper) - set(mut_df.index))
    if missing_genes:
        raise ValueError(
            f"Missing mutation features for {len(missing_genes)} genes. "
            f"Examples: {missing_genes[:10]}"
        )

    x_mut = mut_df.loc[g_upper].values.astype(np.float32)

    if len(x_epi) != len(x_mut) or len(x_epi) != len(y) or len(x_epi) != len(g):
        raise ValueError(
            "Input length mismatch: x_epi, x_mut, y, and g must contain the same number of genes. "
            f"Got x_epi={len(x_epi)}, x_mut={len(x_mut)}, y={len(y)}, g={len(g)}."
        )

    print("Loaded OriGENE-Mut inputs:")
    print(f"  x_epi: {x_epi.shape}")
    print(f"  x_mut: {x_mut.shape}")
    print(f"  y:     {y.shape}")
    print(f"  g:     {g.shape}")

    return x_epi, x_mut, y, g


# <center> SECTION II: OriGENE-Mut MODEL <center>

### <center> 2.1 Helper functions <center>

In [ ]:
########################## Hyperparameters ###############################################

n_splits = 5
batchsize = 53
nE = 100

#########################################################################################


def binary_pred_stats(ytrue, ypred, threshold=0.5):
    """Return sensitivity, specificity, accuracy, and precision."""
    ytrue = np.asarray(ytrue).flatten()
    ypred = np.asarray(ypred).flatten()

    one_correct = np.sum((ytrue == 1) * (ypred > threshold))
    zero_correct = np.sum((ytrue == 0) * (ypred <= threshold))

    sensitivity = one_correct / np.sum(ytrue == 1) if np.sum(ytrue == 1) else 0.0
    specificity = zero_correct / np.sum(ytrue == 0) if np.sum(ytrue == 0) else 0.0
    accuracy = (one_correct + zero_correct) / len(ytrue)
    precision = one_correct / np.sum(ypred > threshold) if np.sum(ypred > threshold) else 0.0

    return sensitivity, specificity, accuracy, precision


def plot_confusion_matrix(
    cm,
    target_names,
    cell_line,
    sample,
    output_dir,
    fold,
    title="Confusion matrix",
    cmap=None,
    normalize=True,
):
    """Plot and save the confusion matrix for one validation fold."""
    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap("Blues")

    plt.figure(figsize=(4, 3))
    plt.imshow(cm, interpolation="nearest", cmap=cmap)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=0, fontsize=12)
        plt.yticks(tick_marks, target_names, fontsize=12)

    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        value = "{:0.4f}".format(cm[i, j]) if normalize else "{:,}".format(cm[i, j])
        plt.text(
            j,
            i,
            value,
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=14,
        )

    plt.tight_layout()
    plt.ylabel("True label", fontsize=14)
    plt.xlabel(
        "Predicted label\naccuracy={:0.4f}; misclass={:0.4f}".format(accuracy, misclass),
        fontsize=14,
    )
    plt.title(title)
    plt.savefig(
        output_dir / f"{cell_line}_{sample}_Fold_{fold}_Confusion.png",
        dpi=50,
        bbox_inches="tight",
    )
    plt.close()


def layerVisualization(model, xtest, ytest, idx, output_dir, xmut=None, layer_type=Conv1D):
    """Visualize filter outputs for a selected validation sequence."""
    outs = [layer.output for layer in model.layers if isinstance(layer, layer_type)]
    intermediate_model = tf.keras.Model(inputs=model.input, outputs=outs)

    if xmut is None:
        dummy_mut = np.zeros((1, model.input[1].shape[1]))
        inputs = [xtest[idx : idx + 1], dummy_mut]
    else:
        inputs = [xtest[idx : idx + 1], xmut[idx : idx + 1]]

    intermediate_outputs = intermediate_model.predict(inputs)
    print("True label:", ytest[idx])

    output_flow = []
    for k, layer in enumerate(intermediate_outputs):
        filters = []
        for ifilter in range(layer.shape[-1]):
            filters.append(layer[0][:, ifilter])
        output_flow.append(filters)

        plt.figure(figsize=(20, 3))
        for filt in filters:
            plt.plot(filt, alpha=0.7)
        plt.title(f"Layer {k + 1} Filters - Sequence Index {idx}")
        plt.xlabel("Position")
        plt.ylabel("Activation")
        plt.tight_layout()
        plt.savefig(output_dir / f"Layer_{k + 1}_Sequence_{idx}_filters.png")
        plt.close()

    return output_flow

### <center> 2.2 Inception module <center>

The inception module follows the original OriGENE model structure and is adapted to one-dimensional epigenetic signal profiles.

In [ ]:
def inception_module(layer_in, f1, f2_in, f2_out, f3_in, f3_out, f4_out):
    """
    Inception module adapted to the 1D OriGENE problem.
    Kernel regularizers are kept as in the original notebook to reduce overfitting.
    """
    reg_value = 1e-2

    conv1 = Conv1D(
        f1,
        kernel_size=1,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(reg_value),
    )(layer_in)

    conv3 = Conv1D(
        f2_in,
        kernel_size=1,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(reg_value),
    )(layer_in)
    conv3 = Conv1D(
        f2_out,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(reg_value),
    )(conv3)

    conv5 = Conv1D(
        f3_in,
        kernel_size=1,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(reg_value),
    )(layer_in)
    conv5 = Conv1D(
        f3_out,
        kernel_size=5,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(reg_value),
    )(conv5)

    pool = MaxPooling1D(pool_size=3, strides=1, padding="same")(layer_in)
    pool = Conv1D(
        f4_out,
        kernel_size=1,
        padding="same",
        activation="relu",
        kernel_regularizer=regularizers.l2(reg_value),
    )(pool)

    return concatenate([conv1, conv3, conv5, pool], axis=-1)

### <center> 2.3 OriGENE-Mut model definition <center>

The model keeps the original OriGENE epigenetic CNN branch. The additional mutation branch transforms 28 mutation-derived features into channel-wise attention weights. These weights modulate the intermediate CNN representation before the second set of inception modules.

This design means that mutation features are not directly concatenated to the final classifier. Instead, they guide the epigenetic representation learned by the CNN.

In [ ]:
def Model_OriGENE_with_mutation(input_shape_epi, input_shape_mut):
    """
    ################################################################################################################
                                    The OriGENE-Mut Model
    ################################################################################################################

    Description:

        OriGENE-Mut consists of I) an epigenetic input section, II) a mutation-feature input section,
        III) a convolutional feature extractor with mutation-guided attention, and IV) a classifier.

        I) The epigenetic input is the same fixed-length OriGENE signal representation obtained from
        the original preprocessing pipeline. Example: x_epi.shape = (None, 40000, 2).

        II) The mutation input is a gene-level vector extracted from VEP-annotated VCF files.
        Example: x_mut.shape = (None, 28). The mutation matrix must follow the same gene order
        as the epigenetic input.

        III) After the first set of inception modules, mutation features are transformed into
        channel-wise attention weights. These weights are repeated across the sequence dimension
        and multiplied with the intermediate CNN representation.

        IV) The classifier integrates the resulting representation into one output: the probability
        that a gene belongs to the cancer driver class (CD) rather than the neutral class (NG).

    _______________________________________________________________________________________________________________
    Inputs. input_shape_epi: Last two indices of the epigenetic array, e.g. (40000, 2)
            input_shape_mut: Last index of the mutation feature matrix, e.g. (28,)

    Outputs. model: Keras model with two inputs and one sigmoid output.
    """

    signal = Input(shape=input_shape_epi, dtype=np.float32, name="epigenetic_signal")
    mutation_input = Input(shape=input_shape_mut, dtype=np.float32, name="mutation_features")

    x = AveragePooling1D(pool_size=20, name="initial_average_pooling")(signal)
    x = Conv1D(4, 3, activation="relu", strides=2, kernel_regularizer=regularizers.l2(1e-2), name="conv1")(x)
    x = Conv1D(8, 3, activation="relu", strides=2, kernel_regularizer=regularizers.l2(1e-2), name="conv2")(x)
    x = AveragePooling1D(pool_size=2, name="average_pooling_after_conv")(x)

    x = inception_module(x, 2, 2, 2, 2, 2, 2)
    x = inception_module(x, 2, 2, 2, 2, 2, 2)
    x = MaxPooling1D(pool_size=2, name="pooling_before_mutation_attention")(x)

    attention_weights = Dense(
        int(x.shape[-1]),
        activation="sigmoid",
        name="mutation_attention_weights",
    )(mutation_input)

    attention_weights = tf.keras.layers.RepeatVector(
        int(x.shape[1]),
        name="repeat_mutation_attention",
    )(attention_weights)

    x = Multiply(name="mutation_guided_attention")([x, attention_weights])

    x = inception_module(x, 4, 4, 4, 4, 4, 4)
    x = inception_module(x, 4, 4, 4, 4, 4, 4)
    x = MaxPooling1D(pool_size=2, name="pooling_after_mutation_attention")(x)
    x = Flatten(name="flatten")(x)

    x = Dropout(0.5, name="dropout_1")(x)
    x = Dense(64, activation="relu", name="dense_64")(x)
    x = Dropout(0.3, name="dropout_2")(x)
    x = Dense(32, activation="relu", name="dense_32")(x)
    x = Dropout(0.2, name="dropout_3")(x)
    output = Dense(1, activation="sigmoid", name="prediction")(x)

    model = Model(inputs=[signal, mutation_input], outputs=output, name="OriGENE_Mut")
    model.compile(
        loss="binary_crossentropy",
        optimizer=Adam(learning_rate=0.0004),
        metrics=["accuracy"],
    )

    print(model.summary())
    return model

### <center> 2.4 Cross-validation training <center>

The training procedure follows the original OriGENE cross-validation design.

Outputs from each fold are saved separately, including predicted probabilities, true labels, gene names, training curves, ROC curves, confusion matrices, and trained model checkpoints.

In [ ]:
def train_oriGENE_crossval_fusion(x_epi, x_mut, y, g, dataset_name, output_dir):
    """
    Train OriGENE-Mut using stratified cross-validation.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=4)
    input_shape_epi = x_epi.shape[1:]
    input_shape_mut = x_mut.shape[1:]

    best_loss = float("inf")
    best_model_path = None

    y_true_global = np.array([])
    y_pred_global = np.array([])
    genes_global = np.array([])

    Avg_performance_metrics = {
        "Sensitivity": 0,
        "Specificity": 0,
        "Accuracy": 0,
        "Precision": 0,
    }
    Avg_AUC = 0

    for fold, (train_idx, val_idx) in enumerate(skf.split(x_epi, y), start=1):
        print(f"\n--- Fold {fold} for {dataset_name} ---")

        x_epi_train, x_epi_val = x_epi[train_idx], x_epi[val_idx]
        x_mut_train, x_mut_val = x_mut[train_idx], x_mut[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        g_val = g[val_idx]

        model = Model_OriGENE_with_mutation(input_shape_epi, input_shape_mut)

        model_file = output_dir / f"OriGENE_Mut_{dataset_name}_Fold{fold}_fusion.keras"
        checkpoint = ModelCheckpoint(str(model_file), monitor="loss", save_best_only=True)

        history = model.fit(
            [x_epi_train, x_mut_train],
            y_train,
            validation_data=([x_epi_val, x_mut_val], y_val),
            epochs=nE,
            batch_size=batchsize,
            shuffle=False,
            callbacks=[checkpoint],
        )

        val_loss = min(history.history["loss"])
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_path = model_file

        y_pred = model.predict([x_epi_val, x_mut_val]).flatten()
        y_true = y_val.flatten()

        y_pred_global = np.append(y_pred_global, y_pred)
        y_true_global = np.append(y_true_global, y_true)
        genes_global = np.append(genes_global, g_val)

        cm = confusion_matrix(y_true, (y_pred > 0.5).astype(int))
        fpr, tpr, _ = roc_curve(y_true, y_pred)
        roc_auc = auc(fpr, tpr)

        sens, spec, acc, prec = binary_pred_stats(y_true, y_pred)
        Avg_performance_metrics["Sensitivity"] += sens / n_splits
        Avg_performance_metrics["Specificity"] += spec / n_splits
        Avg_performance_metrics["Accuracy"] += acc / n_splits
        Avg_performance_metrics["Precision"] += prec / n_splits
        Avg_AUC += roc_auc / n_splits

        print(
            f"{dataset_name} | Fold {fold} | Acc: {acc:.4f}, Sens: {sens:.4f}, "
            f"Spec: {spec:.4f}, Prec: {prec:.4f}, AUC: {roc_auc:.4f}"
        )

        plt.figure(figsize=(4, 4))
        plt.plot(history.history["loss"], label="Loss", color="darkgreen")
        plt.plot(history.history["val_loss"], label="Validation Loss", color="lightgreen")
        plt.plot(history.history["accuracy"], label="Accuracy", color="darkblue")
        plt.plot(history.history["val_accuracy"], label="Validation Accuracy", color="lightblue")
        plt.xlabel("Epochs")
        plt.ylabel("Metric Value")
        plt.legend()
        plt.title(f"Training History - {dataset_name} Fold {fold}")
        plt.savefig(output_dir / f"OriGENE_Mut_{dataset_name}_Fold{fold}_Training.png")
        plt.close()

        plt.figure(figsize=(4, 4))
        plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (area = {roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.legend(loc="lower right")
        plt.title(f"ROC Curve - {dataset_name} Fold {fold}")
        plt.savefig(output_dir / f"OriGENE_Mut_{dataset_name}_Fold{fold}_ROC.png")
        plt.close()

        plot_confusion_matrix(cm, ["NG", "CD"], dataset_name, "OriGENE", output_dir, fold, normalize=False)

        for seq_idx in range(len(val_idx)):
            if val_idx[seq_idx] == 1062:
                print(f"Visualizing filters for sequence index {val_idx[seq_idx]}")
                layerVisualization(model, x_epi_val, y_val, idx=seq_idx, output_dir=output_dir, xmut=x_mut_val)

        np.save(output_dir / f"OriGENE_Mut_{dataset_name}_Fold{fold}_y_pred.npy", y_pred)
        np.save(output_dir / f"OriGENE_Mut_{dataset_name}_Fold{fold}_y_true.npy", y_true)
        np.save(output_dir / f"OriGENE_Mut_{dataset_name}_Fold{fold}_genes.npy", g_val)

    np.save(output_dir / f"OriGENE_Mut_{dataset_name}_all_y_pred.npy", y_pred_global)
    np.save(output_dir / f"OriGENE_Mut_{dataset_name}_all_y_true.npy", y_true_global)
    np.save(output_dir / f"OriGENE_Mut_{dataset_name}_all_genes.npy", genes_global)

    if best_model_path:
        best_model_final_path = output_dir / f"OriGENE_Mut_{dataset_name}_Best_Model.keras"
        shutil.copy(best_model_path, best_model_final_path)
        print(f"Best model saved as: {best_model_final_path}")

    print(f"Average AUC for {dataset_name}: {Avg_AUC:.4f}")
    print("Average metrics:", Avg_performance_metrics)

    return {
        "Average_AUC": Avg_AUC,
        **Avg_performance_metrics,
    }

# <center> SECTION III: RUN MODEL AND SAVE OUTPUTS <center>

### <center> 3.1 Train OriGENE-Mut <center>

Run the following cell after updating the input paths in Section 1.2.

This cell loads the preprocessed OriGENE arrays, aligns mutation features, and trains the mutation-branch model using stratified 5-fold cross-validation.

In [ ]:
x_epi, x_mut, y, g = load_origene_mutation_inputs(
    epigenetic_array_path=EPIGENETIC_ARRAY_PATH,
    label_path=LABEL_PATH,
    gene_path=GENE_PATH,
    mutation_feature_path=MUTATION_FEATURE_PATH,
)

metrics = train_oriGENE_crossval_fusion(
    x_epi=x_epi,
    x_mut=x_mut,
    y=y,
    g=g,
    dataset_name=DATASET_NAME,
    output_dir=OUTPUT_DIR,
)

metrics

### <center> 3.2 Output files <center>

The model outputs are saved to the directory defined by `OUTPUT_DIR`.

Expected output files include:

```text
OriGENE_Mut_<dataset>_Fold<k>_Training.png
OriGENE_Mut_<dataset>_Fold<k>_ROC.png
OriGENE_Mut_<dataset>_Fold_<k>_Confusion.png
OriGENE_Mut_<dataset>_Fold<k>_y_pred.npy
OriGENE_Mut_<dataset>_Fold<k>_y_true.npy
OriGENE_Mut_<dataset>_Fold<k>_genes.npy
OriGENE_Mut_<dataset>_Fold<k>_fusion.keras
OriGENE_Mut_<dataset>_Best_Model.keras
OriGENE_Mut_<dataset>_all_y_pred.npy
OriGENE_Mut_<dataset>_all_y_true.npy
OriGENE_Mut_<dataset>_all_genes.npy
```

# <center> SECTION IV: SUPPLEMENTARY MUTATION FEATURE DOCUMENTATION <center>

### <center> 4.1 Mutation feature overview table <center>

The table below summarizes the mutation-derived features used as the second model input. Detailed formulas, VEP consequence mappings, and pseudocount handling should be reported in the supplementary methods.

| Feature | Description |
|---|---|
| Silent mutations/kb | Density of synonymous mutations |
| Missense mutations/kb | Density of missense mutations |
| LoF mutations/kb | Density of loss-of-function mutations |
| log Total N missense | Log-transformed missense count |
| log Total N LoF | Log-transformed LoF count |
| log Total N splicing | Log-transformed splicing count |
| Missense entropy | Distribution of missense mutations across protein positions |
| Missense/total | Fraction of missense mutations |
| LoF/total | Fraction of LoF mutations |
| Splicing/total | Fraction of splicing mutations |
| Silent fraction | Fraction of silent mutations |
| Nonsense fraction | Fraction of nonsense mutations |
| LoF/Silent ratio | Ratio of LoF to silent mutations |
| Missense/silent ratio | Ratio of missense to silent mutations |
| Splicing/silent ratio | Ratio of splicing to silent mutations |
| LoF/missense ratio | Ratio of LoF to missense mutations |
| Nonsilent/silent ratio | Ratio of nonsilent to silent mutations |
| HiFI/LoFI missense ratio | Damaging vs benign missense ratio |
| LoF/benign ratio | LoF relative to benign mutations |
| Missense/benign ratio | Missense relative to benign mutations |
| Splicing/benign ratio | Splicing relative to benign mutations |
| HiFI missense/benign ratio | Damaging missense relative to benign |
| PolyPhen-2 score | Mean predicted functional impact |
| Recurrent missense fraction | Fraction of missense mutations at recurrent positions |
| Frameshift fraction | Fraction of frameshift mutations |
| Inframe indel fraction | Fraction of inframe insertions/deletions |
| Lost start/stop fraction | Fraction of start/stop lost mutations |
| Inactivating fraction | Fraction of inactivating mutations |

### <center> 4.2 Implementation notes <center>

Mutation-based features are derived from VEP CSQ annotations. Variants are assigned to genes using the `SYMBOL` field, and consequence terms are mapped to predefined mutation categories.

Loss-of-function mutations are defined as the sum of nonsense and frameshift variants. Splicing mutations include splice donor and splice acceptor variants. PolyPhen-2 scores are obtained from the dbNSFP plugin using the HDIV score.

All ratio-based features use a fixed pseudocount of 1 to avoid division by zero. Mutation counts are normalized by coding sequence length in kilobases where applicable.

Recurrent missense fraction is computed as the proportion of missense mutations occurring at protein positions with more than one mutation. If exact reproduction of an earlier exploratory analysis is required, the mutation feature extractor also supports an original mode where this feature is set to zero.

For manuscript reporting, specify which mutation feature mode was used: `original` or `corrected`.